# 🐍 Python & REST API Integration — Lab Notebook
**Trainer: Prachi Kabra**  
**Topics Covered:** Flask · REST APIs · JSON/XML · Authentication · Postman · Data Integration  

---

## How to Use This Notebook
- Run cells **in order** from top to bottom.
- Each Lab section has **objectives**, **code cells**, and **exercises**.
- Cells marked `# ✏️ YOUR CODE HERE` require you to complete them.
- Cells marked `# ✅ SOLUTION` are reference answers — try first before peeking!

### Prerequisites
```bash
pip install flask requests PyJWT xmltodict
```


## Lab 0 — Environment Setup & Verification
> **Objective:** Confirm all dependencies are installed and importable.

In [1]:
# Run this cell first — verify all required packages are available
import sys, importlib

required = ['flask', 'requests', 'jwt', 'xmltodict']
for pkg in required:
    try:
        
        importlib.import_module(pkg)
        print(f'  ✅ {pkg}')
    except ImportError:
        print(f'  ❌ {pkg} — run: pip install {pkg}')

print(f'\nPython version: {sys.version}')


  ✅ flask
  ✅ requests
  ✅ jwt
  ✅ xmltodict

Python version: 3.11.7 | packaged by Anaconda, Inc. | (main, Dec 15 2023, 18:05:47) [MSC v.1916 64 bit (AMD64)]


In [9]:
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return "HOME PAGE WORKING"

@app.route("/test")
def test():
    return "TEST PAGE WORKING"

print(app.url_map)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True)

Map([<Rule '/static/<filename>' (HEAD, GET, OPTIONS) -> static>,
 <Rule '/' (HEAD, GET, OPTIONS) -> home>,
 <Rule '/test' (HEAD, GET, OPTIONS) -> test>])
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\prach\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
!pip install xmltodict


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import threading, time, requests
from flask import Flask, jsonify, request

# Create Flask app
app1 = Flask('lab1')

# ── Route 1: Hello World ──────────────────────────────────────────────
@app1.route('/api/hello', methods=['GET'])
def hello():
    return jsonify({'message': 'Hello, World!', 'status': 'success'}), 200


t1 = threading.Thread(target=run_app1, daemon=True)
t1.start()
time.sleep(1)  # Wait for server to start
print('Flask app1 running on port 5001')

 * Serving Flask app 'lab1'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Flask app1 running on port 5001


127.0.0.1 - - [29/Jun/2026 15:22:02] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:22:02] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:22:17] "GET /api/hello HTTP/1.1" 200 -
127.0.0.1 - - [29/Jun/2026 15:22:52] "GET /api/hello HTTP/1.1" 200 -
127.0.0.1 - - [29/Jun/2026 15:45:25] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:45:40] "GET /api/hello HTTP/1.1" 200 -
127.0.0.1 - - [29/Jun/2026 15:45:51] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:46:44] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:47:16] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:47:27] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:47:36] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:50:50] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 15:51:19] "GET /api/v1/products HTTP/1.1" 404 -


In [10]:
import json 
 
# Python dict → JSON string 
data = {'user': 'Prachi', 'score': 98, 'active': True} 
json_str = json.dumps(data, indent=2) 
print(json_str) 
 
# JSON string → Python dict 
parsed = json.loads(json_str) 
print(parsed['user'])  # 'Prachi' 
 
# Read JSON file 
with open('config.json') as f: 
    config = json.load(f) 
 
# Write JSON file 
with open('output.json', 'w') as f: 
    json.dump(data, f, indent=2) 

{
  "user": "Prachi",
  "score": 98,
  "active": true
}
Prachi


FileNotFoundError: [Errno 2] No such file or directory: 'config.json'

---
## Lab 1 — Flask Basics & Your First API
**Objectives:**
- Create a Flask application
- Define routes with different HTTP methods
- Return JSON responses

> ⚠️ Flask runs a blocking server. We use threading to run it inside Jupyter.

In [2]:
import threading, time, requests
from flask import Flask, jsonify, request

# Create Flask app
app1 = Flask('lab1')

# ── Route 1: Hello World ──────────────────────────────────────────────
@app1.route('/api/hello', methods=['GET'])
def hello():
    return jsonify({'message': 'Hello, World!', 'status': 'success'}), 200

# ── Route 2: Echo — return what was sent ─────────────────────────────
@app1.route('/api/echo', methods=['POST'])
def echo():
    body = request.get_json(force=True)
    return jsonify({'echoed': body}), 200

# ── Route 3: URL parameters ───────────────────────────────────────────
@app1.route('/api/greet/<name>', methods=['GET'])
def greet(name):
    return jsonify({'greeting': f'Hello, {name}!'}), 200

# Start server in background thread
def run_app1():
    app1.run(port=5001, debug=False, use_reloader=False)

t1 = threading.Thread(target=run_app1, daemon=True)
t1.start()
time.sleep(1)  # Wait for server to start
print('Flask app1 running on port 5001')


 * Serving Flask app 'lab1'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Flask app1 running on port 5001


In [4]:



BASE = 'http://localhost:5001'


# GET /api/greet/<name>
r = requests.get(f'{BASE}/api/greet/Learner')
print('GET /api/greet →', r.status_code, r.json())


127.0.0.1 - - [29/Jun/2026 14:44:46] "GET /api/greet/Learner HTTP/1.1" 200 -


GET /api/greet → 200 {'greeting': 'Hello, Learner!'}


In [5]:
# Test the endpoints
BASE = 'http://localhost:5001'

# GET /api/hello
r = requests.get(f'{BASE}/api/hello')
print('GET /api/hello →', r.status_code, r.json())

127.0.0.1 - - [29/Jun/2026 14:44:55] "GET /api/hello HTTP/1.1" 200 -


GET /api/hello → 200 {'message': 'Hello, World!', 'status': 'success'}


127.0.0.1 - - [29/Jun/2026 14:47:15] "GET /api/hello HTTP/1.1" 200 -
127.0.0.1 - - [29/Jun/2026 14:47:31] "POST /api/echo HTTP/1.1" 200 -


In [32]:
# POST /api/echo
import requests

BASE = "http://localhost:5001"

response = requests.post(
    f"{BASE}/api/echo",
    json={
        "name": "Prachi",
        "course": "Flask API"
    }
)

print(response.status_code)
print(response.json())

127.0.0.1 - - [29/Jun/2026 13:51:40] "POST /api/echo HTTP/1.1" 200 -


200
{'echoed': {'course': 'Flask API', 'name': 'Prachi'}}


127.0.0.1 - - [29/Jun/2026 13:52:02] "GET /api/echo HTTP/1.1" 405 -
127.0.0.1 - - [29/Jun/2026 13:53:22] "GET /api/greet/Anand HTTP/1.1" 200 -


### ✏️ Exercise 1.1
Add a new route `/api/add` that accepts `a` and `b` as **query parameters** and returns their sum.

Example: `GET /api/add?a=5&b=3` → `{'result': 8}`

In [10]:
# ✏️ YOUR CODE HERE
# Hint: use request.args.get('a', 0, type=float)

@app1.route('/api/add', methods=['GET'])
def add():
    # TODO: get 'a' and 'b' from query params, return their sum
    pass


In [11]:
# ✅ SOLUTION — uncomment to see
# @app1.route('/api/add', methods=['GET'])
# def add():
#     a = request.args.get('a', 0, type=float)
#     b = request.args.get('b', 0, type=float)
#     return jsonify({'result': a + b}), 200

# r = requests.get(f'{BASE}/api/add?a=5&b=3')
# print(r.json())  # {'result': 8.0}


---
## Lab 2 — Full CRUD REST API with Error Handling
**Objectives:**
- Implement GET, POST, PUT, DELETE endpoints
- Handle 404 and 400 errors gracefully
- Use Blueprints for organisation

In [12]:
import threading, time, requests
from flask import Flask, jsonify, request, abort, Blueprint

app2 = Flask('lab2')

# ── In-memory store ───────────────────────────────────────────────────
products = {}
next_id = [1]  # Use list to allow mutation inside functions

# ── Error handlers ────────────────────────────────────────────────────
@app2.errorhandler(400)
def bad_request(e):
    return jsonify({'error': 'Bad Request', 'message': str(e)}), 400

@app2.errorhandler(404)
def not_found(e):
    return jsonify({'error': 'Not Found', 'message': str(e)}), 404

# ── Endpoints ─────────────────────────────────────────────────────────
@app2.route('/api/v1/products', methods=['GET'])
def get_products():
    return jsonify({'data': list(products.values()), 'count': len(products)}), 200

@app2.route('/api/v1/products/<int:pid>', methods=['GET'])
def get_product(pid):
    p = products.get(pid)
    if not p:
        abort(404, description=f'Product {pid} not found')
    return jsonify({'data': p}), 200

@app2.route('/api/v1/products', methods=['POST'])
def create_product():
    body = request.get_json(force=True)
    if not body or 'name' not in body:
        abort(400, description='Field "name" is required')
    pid = next_id[0]
    product = {'id': pid, 'name': body['name'], 'price': float(body.get('price', 0))}
    products[pid] = product
    next_id[0] += 1
    return jsonify({'data': product}), 201

@app2.route('/api/v1/products/<int:pid>', methods=['PUT'])
def update_product(pid):
    if pid not in products:
        abort(404, description=f'Product {pid} not found')
    body = request.get_json(force=True)
    if 'name' in body:
        products[pid]['name'] = body['name']
    if 'price' in body:
        products[pid]['price'] = float(body['price'])
    return jsonify({'data': products[pid]}), 200

@app2.route('/api/v1/products/<int:pid>', methods=['DELETE'])
def delete_product(pid):
    if pid not in products:
        abort(404)
    del products[pid]
    return '', 204

# Start server
t2 = threading.Thread(target=lambda: app2.run(port=5002, debug=False, use_reloader=False), daemon=True)
t2.start()
time.sleep(1)
print('Flask app2 (CRUD) running on port 5002')


 * Serving Flask app 'lab2'
 * Debug mode: off


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit


Flask app2 (CRUD) running on port 5002


127.0.0.1 - - [29/Jun/2026 13:40:10] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 13:40:11] "GET /favicon.ico HTTP/1.1" 404 -


In [13]:
BASE2 = 'http://localhost:5002/api/v1'

# CREATE
r = requests.post(f'{BASE2}/products', json={'name': 'Laptop', 'price': 1299.99})
print('POST (create) →', r.status_code, r.json())
laptop_id = r.json()['data']['id']

# CREATE another
r = requests.post(f'{BASE2}/products', json={'name': 'Mouse', 'price': 25.00})
print('POST (create) →', r.status_code, r.json())

# READ all
r = requests.get(f'{BASE2}/products')
print('GET (all)     →', r.status_code, r.json())

# READ one
r = requests.get(f'{BASE2}/products/{laptop_id}')
print('GET (one)     →', r.status_code, r.json())

# UPDATE
r = requests.put(f'{BASE2}/products/{laptop_id}', json={'price': 999.99})
print('PUT (update)  →', r.status_code, r.json())

# DELETE
r = requests.delete(f'{BASE2}/products/{laptop_id}')
print('DELETE        →', r.status_code)

# Verify 404
r = requests.get(f'{BASE2}/products/{laptop_id}')
print('GET (deleted) →', r.status_code, r.json())


127.0.0.1 - - [29/Jun/2026 13:40:26] "POST /api/v1/products HTTP/1.1" 201 -


POST (create) → 201 {'data': {'id': 1, 'name': 'Laptop', 'price': 1299.99}}


127.0.0.1 - - [29/Jun/2026 13:40:28] "POST /api/v1/products HTTP/1.1" 201 -


POST (create) → 201 {'data': {'id': 2, 'name': 'Mouse', 'price': 25.0}}


127.0.0.1 - - [29/Jun/2026 13:40:30] "GET /api/v1/products HTTP/1.1" 200 -


...

127.0.0.1 - - [29/Jun/2026 13:40:36] "DELETE /api/v1/products/1 HTTP/1.1" 204 -


DELETE        → 204


127.0.0.1 - - [29/Jun/2026 13:40:38] "GET /api/v1/products/1 HTTP/1.1" 404 -


GET (deleted) → 404 {'error': 'Not Found', 'message': '404 Not Found: Product 1 not found'}


127.0.0.1 - - [29/Jun/2026 13:40:46] "GET /api/v1 HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 13:40:46] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [29/Jun/2026 13:41:10] "GET /products HTTP/1.1" 404 -


### ✏️ Exercise 2.1 — PATCH endpoint
Add a `PATCH /api/v1/products/<id>` endpoint to `app2` that only updates fields provided in the request body (partial update). Test it with Postman or the requests library.

### ✏️ Exercise 2.2 — Validation
Modify the `create_product` function to also validate that `price` is a positive number. Return a 400 error if it is not.

---
## Lab 3 — JSON & XML Serialisation
**Objectives:**
- Parse and produce JSON and XML
- Convert between the two formats
- Serve XML from Flask


In [14]:
import json

# ── JSON: encode / decode ─────────────────────────────────────────────
data = {
    'company': 'TechCorp',
    'employees': [
        {'id': 1, 'name': 'Alice', 'role': 'Engineer', 'active': True},
        {'id': 2, 'name': 'Bob',   'role': 'Manager',  'active': False},
    ],
    'revenue_usd': 5_000_000.00
}

# Serialise
json_str = json.dumps(data, indent=2)
print('--- JSON Output ---')
print(json_str)

# De-serialise
parsed = json.loads(json_str)
active_employees = [e['name'] for e in parsed['employees'] if e['active']]
print('\nActive employees:', active_employees)


--- JSON Output ---
{
  "company": "TechCorp",
  "employees": [
    {
      "id": 1,
      "name": "Alice",
      "role": "Engineer",
      "active": true
    },
    {
      "id": 2,
      "name": "Bob",
      "role": "Manager",
      "active": false
    }
  ],
  "revenue_usd": 5000000.0
}

Active employees: ['Alice']


In [15]:
import xml.etree.ElementTree as ET

# ── Build XML ─────────────────────────────────────────────────────────
root = ET.Element('catalog')

items = [('101', 'Laptop', 'USD', '1299.99'), ('102', 'Mouse', 'USD', '25.00')]
for pid, name, currency, price in items:
    product = ET.SubElement(root, 'product', id=pid)
    ET.SubElement(product, 'name').text = name
    price_el = ET.SubElement(product, 'price')
    price_el.set('currency', currency)
    price_el.text = price

ET.indent(root, space='  ')
xml_str = ET.tostring(root, encoding='unicode')
print('--- XML Output ---')
print(xml_str)


--- XML Output ---
<catalog>
  <product id="101">
    <name>Laptop</name>
    <price currency="USD">1299.99</price>
  </product>
  <product id="102">
    <name>Mouse</name>
    <price currency="USD">25.00</price>
  </product>
</catalog>


In [16]:
# ── Parse XML ─────────────────────────────────────────────────────────
parsed_root = ET.fromstring(xml_str)
print('Parsed products:')
for product in parsed_root.findall('product'):
    pid   = product.get('id')
    name  = product.find('name').text
    price = product.find('price').text
    curr  = product.find('price').get('currency')
    print(f'  [{pid}] {name} — {curr} {price}')


Parsed products:
  [101] Laptop — USD 1299.99
  [102] Mouse — USD 25.00


In [17]:
import xmltodict

# ── XML → Python dict → JSON ──────────────────────────────────────────
as_dict = xmltodict.parse(xml_str)
as_json = json.dumps(as_dict, indent=2)
print('--- XML → JSON ---')
print(as_json)

# ── JSON dict → XML ───────────────────────────────────────────────────
data_for_xml = {'products': {'product': [
    {'@id': '201', 'name': 'Keyboard', 'price': {'@currency': 'USD', '#text': '75.00'}}
]}}
back_to_xml = xmltodict.unparse(data_for_xml, pretty=True)
print('\n--- JSON → XML ---')
print(back_to_xml)


--- XML → JSON ---
{
  "catalog": {
    "product": [
      {
        "@id": "101",
        "name": "Laptop",
        "price": {
          "@currency": "USD",
          "#text": "1299.99"
        }
      },
      {
        "@id": "102",
        "name": "Mouse",
        "price": {
          "@currency": "USD",
          "#text": "25.00"
        }
      }
    ]
  }
}

--- JSON → XML ---
<?xml version="1.0" encoding="utf-8"?>
<products>
	<product id="201">
		<name>Keyboard</name>
		<price currency="USD">75.00</price>
	</product>
</products>


### ✏️ Exercise 3.1
Write a function `xml_to_csv(xml_string)` that:
1. Parses the catalog XML
2. Returns a CSV string with columns: `id,name,currency,price`

Expected output:
```
id,name,currency,price
101,Laptop,USD,1299.99
102,Mouse,USD,25.00
```

In [18]:
# ✏️ YOUR CODE HERE
def xml_to_csv(xml_string: str) -> str:
    """
    Parse XML catalog and return CSV string.
    Columns: id,name,currency,price
    """
    # TODO: parse xml_string, iterate products, build CSV
    pass

print(xml_to_csv(xml_str))


None


In [ ]:
# ✅ SOLUTION
# def xml_to_csv(xml_string):
#     import io
#     root = ET.fromstring(xml_string)
#     lines = ['id,name,currency,price']
#     for p in root.findall('product'):
#         pid  = p.get('id')
#         name = p.find('name').text
#         curr = p.find('price').get('currency')
#         price = p.find('price').text
#         lines.append(f'{pid},{name},{curr},{price}')
#     return '\n'.join(lines)

# print(xml_to_csv(xml_str))


---
## Lab 4 — API Authentication: API Key & JWT
**Objectives:**
- Implement API Key authentication as a decorator
- Issue and validate JWT tokens
- Protect endpoints with JWT


In [19]:
import threading, time, jwt, datetime, requests
from flask import Flask, jsonify, request, abort
from functools import wraps

app4 = Flask('lab4')
SECRET_KEY = 'lab4-jwt-secret-change-in-production'
VALID_API_KEYS = {'key-abc-123', 'key-xyz-789'}

# ── Decorator: API Key ────────────────────────────────────────────────
def require_api_key(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        key = request.headers.get('X-API-Key', '')
        if key not in VALID_API_KEYS:
            return jsonify({'error': 'Unauthorized', 'message': 'Invalid API key'}), 401
        return f(*args, **kwargs)
    return wrapper

# ── Decorator: JWT ────────────────────────────────────────────────────
def jwt_required(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        auth = request.headers.get('Authorization', '')
        if not auth.startswith('Bearer '):
            return jsonify({'error': 'Unauthorized', 'message': 'Missing Bearer token'}), 401
        token = auth[7:]
        try:
            payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
            request.current_user = payload['sub']
        except jwt.ExpiredSignatureError:
            return jsonify({'error': 'Unauthorized', 'message': 'Token expired'}), 401
        except jwt.InvalidTokenError:
            return jsonify({'error': 'Unauthorized', 'message': 'Invalid token'}), 401
        return f(*args, **kwargs)
    return wrapper

# ── Endpoints ─────────────────────────────────────────────────────────
@app4.route('/auth/login', methods=['POST'])
def login():
    body = request.get_json(force=True)
    USERS = {'admin': 'pass123', 'prachi': 'flask456'}
    username = body.get('username', '')
    password = body.get('password', '')
    if USERS.get(username) != password:
        return jsonify({'error': 'Invalid credentials'}), 401
    payload = {
        'sub': username,
        'iat': datetime.datetime.utcnow(),
        'exp': datetime.datetime.utcnow() + datetime.timedelta(hours=1)
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm='HS256')
    return jsonify({'token': token, 'expires_in': 3600}), 200

@app4.route('/api/public')
def public_route():
    return jsonify({'message': 'This is public — no auth needed'}), 200

@app4.route('/api/apikey-protected')
@require_api_key
def apikey_protected():
    return jsonify({'message': 'Accessed via API Key ✅'}), 200

@app4.route('/api/jwt-protected')
@jwt_required
def jwt_protected():
    return jsonify({'message': f'Hello, {request.current_user}! JWT verified ✅'}), 200

t4 = threading.Thread(target=lambda: app4.run(port=5004, debug=False, use_reloader=False), daemon=True)
t4.start()
time.sleep(1)
print('Flask app4 (auth) running on port 5004')


 * Serving Flask app 'lab4'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit


Flask app4 (auth) running on port 5004


In [20]:
BASE4 = 'http://localhost:5004'

# 1. Public endpoint
r = requests.get(f'{BASE4}/api/public')
print('Public          →', r.status_code, r.json())

# 2. API Key — valid
r = requests.get(f'{BASE4}/api/apikey-protected', headers={'X-API-Key': 'key-abc-123'})
print('API Key (valid)  →', r.status_code, r.json())

# 3. API Key — invalid
r = requests.get(f'{BASE4}/api/apikey-protected', headers={'X-API-Key': 'wrong-key'})
print('API Key (bad)    →', r.status_code, r.json())

# 4. Login to get JWT
r = requests.post(f'{BASE4}/auth/login', json={'username': 'prachi', 'password': 'flask456'})
print('Login            →', r.status_code)
token = r.json().get('token')
print('Token received   :', token[:40], '...')

# 5. JWT-protected endpoint
r = requests.get(f'{BASE4}/api/jwt-protected', headers={'Authorization': f'Bearer {token}'})
print('JWT (valid)      →', r.status_code, r.json())

# 6. JWT-protected with bad token
r = requests.get(f'{BASE4}/api/jwt-protected', headers={'Authorization': 'Bearer bad.token.here'})
print('JWT (bad)        →', r.status_code, r.json())


127.0.0.1 - - [29/Jun/2026 13:41:46] "GET /api/public HTTP/1.1" 200 -


Public          → 200 {'message': 'This is public — no auth needed'}


127.0.0.1 - - [29/Jun/2026 13:41:48] "GET /api/apikey-protected HTTP/1.1" 200 -


API Key (valid)  → 200 {'message': 'Accessed via API Key ✅'}


127.0.0.1 - - [29/Jun/2026 13:41:50] "GET /api/apikey-protected HTTP/1.1" 401 -


...

Login            → 200
Token received   : eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ ...


127.0.0.1 - - [29/Jun/2026 13:41:54] "GET /api/jwt-protected HTTP/1.1" 200 -


JWT (valid)      → 200 {'message': 'Hello, prachi! JWT verified ✅'}


127.0.0.1 - - [29/Jun/2026 13:41:57] "GET /api/jwt-protected HTTP/1.1" 401 -


JWT (bad)        → 401 {'error': 'Unauthorized', 'message': 'Invalid token'}


### ✏️ Exercise 4.1 — Decode and Inspect a JWT
Using the `token` variable from the cell above:
1. Decode it using `jwt.decode()` to inspect the payload claims
2. Print: `sub`, `iat` (as human-readable datetime), `exp` (as human-readable datetime)


In [ ]:
# ✏️ YOUR CODE HERE
# Hint: jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
# Hint: datetime.datetime.utcfromtimestamp(timestamp)

SECRET_KEY = 'lab4-jwt-secret-change-in-production'
# TODO: decode and print the payload


---
## Lab 5 — Consuming External APIs & Pagination
**Objectives:**
- Use the `requests` library to call public APIs
- Handle pagination
- Build a resilient session with retry logic


In [21]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ── Resilient session ─────────────────────────────────────────────────
def build_session(retries: int = 3, backoff: float = 0.5) -> requests.Session:
    """
    Creates a requests Session with automatic retry and exponential backoff.
    Retries on: 429 (rate limit), 500, 502, 503, 504.
    """
    session = requests.Session()
    retry = Retry(
        total=retries,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET', 'POST', 'PUT']
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

session = build_session()
print('Resilient session created ✅')


Resilient session created ✅


In [22]:
# ── Public API: JSONPlaceholder ───────────────────────────────────────
# JSONPlaceholder is a free fake REST API for testing

# GET all posts (simulates paginated API)
resp = session.get('https://jsonplaceholder.typicode.com/posts', timeout=10)
resp.raise_for_status()
all_posts = resp.json()
print(f'Total posts fetched: {len(all_posts)}')
print('First post:', all_posts[0])


Total posts fetched: 100
First post: {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [23]:
# ── Pagination simulation ─────────────────────────────────────────────
def fetch_paginated(session, url: str, page_size: int = 10) -> list:
    """
    Simulates offset-based pagination.
    JSONPlaceholder uses _start and _limit params.
    """
    results = []
    start = 0
    while True:
        resp = session.get(url, params={'_start': start, '_limit': page_size}, timeout=10)
        resp.raise_for_status()
        page = resp.json()
        if not page:
            break
        results.extend(page)
        print(f'  Fetched records {start}–{start + len(page) - 1}')
        if len(page) < page_size:
            break  # Last page
        start += page_size
    return results

posts = fetch_paginated(session, 'https://jsonplaceholder.typicode.com/posts', page_size=20)
print(f'\nTotal records collected: {len(posts)}')


  Fetched records 0–19
  Fetched records 20–39
  Fetched records 40–59
  Fetched records 60–79
  Fetched records 80–99

Total records collected: 100


In [24]:
# ── Error handling patterns ───────────────────────────────────────────
import requests

def safe_get(url: str, **kwargs) -> dict | None:
    """Wraps GET with full error handling, returns None on failure."""
    try:
        resp = session.get(url, timeout=10, **kwargs)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.Timeout:
        print(f'  ⏱ Timeout for {url}')
    except requests.exceptions.ConnectionError:
        print(f'  🔌 Connection error for {url}')
    except requests.exceptions.HTTPError as e:
        print(f'  ❌ HTTP {e.response.status_code} for {url}: {e}')
    except ValueError:
        print(f'  📄 JSON parse error for {url}')
    return None

# Test with valid URL
result = safe_get('https://jsonplaceholder.typicode.com/users/1')
if result:
    print('User:', result.get('name'), '|', result.get('email'))

# Test with 404 URL
result = safe_get('https://jsonplaceholder.typicode.com/users/9999')
print('404 result:', result)


User: Leanne Graham | Sincere@april.biz
  ❌ HTTP 404 for https://jsonplaceholder.typicode.com/users/9999: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/users/9999
404 result: None


---
## Lab 6 — Data Integration Capstone
**Objectives:**
- Pull data from your Flask CRUD API
- Enrich with a public API
- Transform and save as JSON

> Make sure Lab 2's app (port 5002) is still running from earlier cells.

In [25]:
import requests, json
from datetime import datetime, timezone

CRUD_BASE = 'http://localhost:5002/api/v1'
FX_URL    = 'https://open.er-api.com/v6/latest/USD'

# Step 1: Seed some products into the CRUD API
seed_products = [
    {'name': 'Laptop Pro',   'price': 1499.00},
    {'name': 'Wireless Mouse', 'price': 39.99},
    {'name': 'USB-C Hub',    'price': 55.00},
    {'name': 'Mechanical Keyboard', 'price': 89.99},
]

for p in seed_products:
    r = requests.post(f'{CRUD_BASE}/products', json=p)
    print(f'  Created: [{r.status_code}] {r.json()["data"]["name"]}')


127.0.0.1 - - [29/Jun/2026 13:42:30] "POST /api/v1/products HTTP/1.1" 201 -


  Created: [201] Laptop Pro


127.0.0.1 - - [29/Jun/2026 13:42:32] "POST /api/v1/products HTTP/1.1" 201 -


  Created: [201] Wireless Mouse


127.0.0.1 - - [29/Jun/2026 13:42:34] "POST /api/v1/products HTTP/1.1" 201 -


  Created: [201] USB-C Hub


127.0.0.1 - - [29/Jun/2026 13:42:36] "POST /api/v1/products HTTP/1.1" 201 -


  Created: [201] Mechanical Keyboard


In [26]:
# Step 2: Fetch all products
r = requests.get(f'{CRUD_BASE}/products')
r.raise_for_status()
products_raw = r.json()['data']
print(f'Fetched {len(products_raw)} products from Flask API')


127.0.0.1 - - [29/Jun/2026 13:42:38] "GET /api/v1/products HTTP/1.1" 200 -


Fetched 5 products from Flask API


In [27]:
# Step 3: Fetch FX rates
try:
    r = requests.get(FX_URL, timeout=8)
    r.raise_for_status()
    rates = r.json().get('rates', {})
    print(f'FX rates loaded. USD → INR: {rates.get("INR", "N/A")}')
except Exception as e:
    print(f'FX API unavailable ({e}), using fallback rates')
    rates = {'INR': 83.5, 'EUR': 0.92, 'GBP': 0.79}  # Fallback


FX rates loaded. USD → INR: 94.490194


In [28]:
# Step 4: Enrich products with multi-currency prices
def enrich_product(product: dict, rates: dict, targets: list = ['INR', 'EUR', 'GBP']) -> dict:
    enriched = dict(product)
    usd_price = product['price']
    enriched['prices'] = {'USD': round(usd_price, 2)}
    for currency in targets:
        rate = rates.get(currency)
        if rate:
            enriched['prices'][currency] = round(usd_price * rate, 2)
    enriched['enriched_at'] = datetime.now(timezone.utc).isoformat()
    return enriched

enriched_products = [enrich_product(p, rates) for p in products_raw]

# Preview
for ep in enriched_products:
    print(f"  {ep['name']:25s} USD: {ep['prices']['USD']:8.2f} | "
          f"INR: {ep['prices'].get('INR', 'N/A'):10} | "
          f"EUR: {ep['prices'].get('EUR', 'N/A')}")


  Mouse                     USD:    25.00 | INR:    2362.25 | EUR: 21.96
  Laptop Pro                USD:  1499.00 | INR:   141640.8 | EUR: 1316.49
  Wireless Mouse            USD:    39.99 | INR:    3778.66 | EUR: 35.12
  USB-C Hub                 USD:    55.00 | INR:    5196.96 | EUR: 48.3
  Mechanical Keyboard       USD:    89.99 | INR:    8503.17 | EUR: 79.03


In [29]:
# Step 5: Save enriched data to JSON file
output = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_api': CRUD_BASE,
    'fx_source': FX_URL,
    'record_count': len(enriched_products),
    'data': enriched_products
}

with open('enriched_products.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f'✅ Saved {len(enriched_products)} enriched products to enriched_products.json')

# Quick verify
with open('enriched_products.json') as f:
    loaded = json.load(f)
print(f'Verification: file has {loaded["record_count"]} records')


✅ Saved 5 enriched products to enriched_products.json
Verification: file has 5 records


### ✏️ Capstone Exercise
Extend the pipeline to also:
1. Generate an **XML report** of enriched products (use `xml.etree.ElementTree`)
2. Add a `category` field — classify products as `'Electronics'` if price > 100, else `'Accessories'`
3. Save both `enriched_products.json` and `enriched_products.xml`


---
## Lab 7 — Postman Testing Guide
> This lab provides Postman test scripts to copy-paste. No code execution needed here.

### 7.1 Environment Variables
Create a Postman Environment called **Dev** with:
| Variable | Value |
|---|---|
| `BASE_URL` | `http://localhost:5002` |
| `API_KEY` | `key-dev-999` |
| `jwt_token` | *(empty — auto-populated by login script)* |

### 7.2 Test Scripts
Add these to the **Tests** tab of each request:

**POST /api/v1/products — Create Product**
```javascript
pm.test('Status is 201', () => pm.response.to.have.status(201));
pm.test('Response has id', () => {
    const body = pm.response.json();
    pm.expect(body.data).to.have.property('id');
    pm.environment.set('product_id', body.data.id);
});
pm.test('Name matches input', () => {
    pm.expect(pm.response.json().data.name).to.eql(pm.request.body.raw ? JSON.parse(pm.request.body.raw).name : '');
});
```

**GET /api/v1/products — List Products**
```javascript
pm.test('Status is 200', () => pm.response.to.have.status(200));
pm.test('Data is array', () => pm.expect(pm.response.json().data).to.be.an('array'));
pm.test('Count matches data length', () => {
    const body = pm.response.json();
    pm.expect(body.count).to.eql(body.data.length);
});
```

### 7.3 Pre-request Script — Auto JWT Login
Add to collection level Pre-request Scripts:
```javascript
const token = pm.environment.get('jwt_token');
if (!token) {
    pm.sendRequest({
        url: pm.environment.get('BASE_URL') + '/auth/login',
        method: 'POST',
        header: {'Content-Type': 'application/json'},
        body: { mode: 'raw', raw: JSON.stringify({username:'admin',password:'pass123'}) }
    }, (err, res) => {
        if (!err) pm.environment.set('jwt_token', res.json().token);
    });
}
```


---
## 🏁 Summary & Next Steps

### What We Built
| Lab | Achievement |
|-----|-------------|
| Lab 1 | First Flask API with routing and responses |
| Lab 2 | Full CRUD API with error handling |
| Lab 3 | JSON & XML serialisation and conversion |
| Lab 4 | API Key and JWT authentication |
| Lab 5 | Consuming external APIs with pagination & retries |
| Lab 6 | End-to-end data integration pipeline |
| Lab 7 | Postman collection with automated tests |

### Recommended Next Steps
- **Flask-SQLAlchemy** — replace in-memory store with a real database
- **Marshmallow** — schema validation and serialisation
- **Flask-Migrate** — database migrations
- **Swagger/OpenAPI** — auto-generate API documentation with `flask-smorest`
- **Docker** — containerise your Flask API
- **FastAPI** — async alternative to Flask for high-performance APIs

---
*Trainer: Prachi Kabra*
